# modeling_v14 — V0 시간축 하니스 + 벤치마크 self-check

**로드맵 단계** V0 · **성격** 하니스(개발 프레임) · **판정 축** rolling-origin time-CV(主) · **공식 판정** = 강건(회신 숫자)

## 이 노트북이 하는 일
1. **rolling-origin time-CV 프레임** — 확장창 · 초기 4주 · horizon 7일 · fold마다 재학습(인과적, 과거→단기미래)
2. **벤치마크 self-check** — 3-arm(재학습 core10 / 재학습 lean-85 / 무재학습) 로 REPORT_13 재현 (core10 = **pooled 98.36 · R² 0.85**)
3. **same-era 로더** — valid/test 원시 트레이스 로드·시각파싱 확인 (**채점은 V4 부록까지 보류**, R3)
4. **P1~P4 피처 빌더 스켈레톤** + **R11 인과 assert 자기검증** (생성은 V1)

## 전제 파일 (경로 자동탐색 `_find`)
- `v13_fdc_pool_wf_oof.csv.gz` (11939×655 웨이퍼 집계풀) — `../modeling_v13/data/`
- `core10_meta_wf.csv` (core10 레짐/시간 8피처) — `../modeling_v13/colab_GA/`
- `lean_timestable_set.json` · `tuned_params_v13_xgb.json` — `../modeling_v13/data/`
- `train_data.csv` (C40 시각원본) — `../문제1(하)/`
- valid/test: `../문제1(하)/valid_X.csv` · `test_X.csv`

## 환경 핀 (R6 — 공식 수치는 로컬 venv 산만)
python 3.12.10 · xgboost **3.3.0** · lightgbm 4.6.0 · sklearn 1.9.0 · numpy 2.5.1 · pandas 3.0.3

## 예상 런타임
CPU 로컬 venv 기준 **약 1~3분** (3-arm × 약 6 주간 fold, XGB 246r depth4).

## 확인 포인트 (GV0 — 강건 판정용)
- [ ] **재학습 core10 pooled ≈ 98.36 · R² ≈ 0.85** 재현 (±소폭) → 하니스 신뢰(R4 GV0)
- [ ] 구조 재현: 재학습 ≪ 무재학습(≈255) · 센서 lean-85(≈99.8) **>** core10(≈98.4)
- [ ] `[셀3b]` **R11 인과 assert PASS** + 반례(center) 누수 탐지=True
- [ ] `[셀3]` floor 5센서(C17·C11·C31·C15·C16) 각 ≥1 · R2 누수 assert 통과
- [ ] `[셀6]` valid 시각 파싱 O · test 파싱 X(PLAN 확증) · 정답폴더 **경로만**(미접근)

> **미러(참고)**: 클라우드 xgb 2.1.4 에서 core10 **103.6**(R²0.843) — 환경 +5pt 오프셋, 구조 동일. 절대 98.36 은 **로컬 확정**(R6).


In [1]:
# [셀1] import + 동결 상수
import warnings; warnings.filterwarnings("ignore")
import os, re, json, time
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_squared_error

ID_COL, TARGET_COL, C20_COL = "C64", "C65", "C20"
FMT = "%Y-%m-%d %H:%M:%S.%f"
# core10 = 레짐/시간7 + 센서/레시피3 (modeling_v8 M4 동결)
CORE10 = ["is_high_regime", "high_regime_days", "days_since_last_pm", "C33",
          "dslp_x_hour", "hour", "hour_x_c33", "C60_mean_step4", "C59_mean_step4", "is_special_recipe"]
PROTECTED = ["C17", "C11", "C31", "C15", "C16"]   # floor 필수 5센서 (R10)
SIGMA_C65 = 261.7                                  # C65 전역 std(ddof=0) — honest R² 분모(REPORT_13 동일)

# rolling-origin 프로토콜 상수 (동결 — b2prime 승계)
INIT_WEEKS = 4        # 초기 학습창
H_DAYS = 7            # horizon (주간 fold)
MIN_TEST = 30         # 주간 fold 최소 test 웨이퍼

XGB_DEVICE = os.environ.get("XGB_DEVICE", "cpu")

def _find(name):
    for d in [".", "data", "colab_GA", "..", os.path.join("..", "modeling_v13"),
              os.path.join("..", "modeling_v13", "data"), os.path.join("..", "modeling_v13", "colab_GA"),
              os.path.join("..", "문제1(하)"), "문제1(하)"]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} 없음")

POOL = _find("v13_fdc_pool_wf_oof.csv.gz"); META = _find("core10_meta_wf.csv")
LEANJSON = _find("lean_timestable_set.json"); XGBJSON = _find("tuned_params_v13_xgb.json"); TRAIN = _find("train_data.csv")
print("xgboost", xgb.__version__, "| device", XGB_DEVICE, "| INIT", INIT_WEEKS, "wk | H", H_DAYS, "d | MIN_TEST", MIN_TEST)

xgboost 3.3.0 | device cpu | INIT 4 wk | H 7 d | MIN_TEST 30


In [2]:
# [셀2] 헬퍼
def _rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

def sensor_of(c):
    m = re.match(r"(C\d+)_", c); return m.group(1) if m else c

def floor_ok(feats):
    have = {s: sum(1 for c in feats if sensor_of(c) == s) for s in PROTECTED}
    return all(v >= 1 for v in have.values()), have

def r2_honest(rmse):
    return round(1 - (rmse / SIGMA_C65) ** 2, 4)

def make_xgb(params, rounds):
    p = dict(params)
    p.update(objective="reg:squarederror", tree_method="hist", device=XGB_DEVICE,
             random_state=42, n_estimators=int(rounds))
    return xgb.XGBRegressor(**p)

In [3]:
# [셀3] 로드 + core10 병합 + 웨이퍼/Lot 시간 + asserts (R2 누수 · R10 floor)
pool = pd.read_csv(POOL); pool[ID_COL] = pool[ID_COL].astype(str)
meta = pd.read_csv(META); meta[ID_COL] = meta[ID_COL].astype(str)
df = pool.merge(meta, on=ID_COL, how="inner")

# 웨이퍼 시각 = train C40 min per 웨이퍼 (인과 정렬 기준)
tt = pd.read_csv(TRAIN, usecols=[ID_COL, "C40"]); tt[ID_COL] = tt[ID_COL].astype(str)
tt["_ts"] = pd.to_datetime(tt["C40"], format=FMT)
df = df.merge(tt.groupby(ID_COL)["_ts"].min().reset_index().rename(columns={"_ts": "wf_ts"}),
              on=ID_COL, how="inner").reset_index(drop=True)
y = df[TARGET_COL].to_numpy(float)

LEAN = json.load(open(LEANJSON, encoding="utf-8"))["lean_features"]
CORE = [f for f in CORE10 if f in df.columns]
xgj = json.load(open(XGBJSON, encoding="utf-8")); XGB_PARAMS = xgj["params"]; XGB_ROUNDS = int(xgj["n_estimators"])

ok, have = floor_ok(LEAN)
assert all(f in df.columns for f in LEAN), "lean 누락 피처"
assert C20_COL not in LEAN and ID_COL not in LEAN and "fold_kf5" not in LEAN, "R2 위반: ID 피처 유입"
assert C20_COL not in CORE and ID_COL not in CORE, "R2 위반: core 에 ID"
assert ok, f"R10 floor 위반 {have}"
assert len(CORE) == 10, f"core10 != 10 ({len(CORE)})"

# Lot 대표시각 = Lot 내 웨이퍼 median → Lot 단위 인과 정렬
lot_time = df.groupby(C20_COL)["wf_ts"].median()
df["lot_ts"] = df[C20_COL].map(lot_time)
print(f"[셀3] df {df.shape} | lean {len(LEAN)} floor={have} | core10 {len(CORE)} | "
      f"기간 {str(df['wf_ts'].min())[:10]}~{str(df['wf_ts'].max())[:10]} | XGB {XGB_ROUNDS}r")

[셀3] df (11939, 669) | lean 85 floor={'C17': 2, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 1} | core10 10 | 기간 2018-12-01~2019-02-08 | XGB 246r


In [4]:
# [셀3b] R11 인과 assert 자기검증 — P3형 trailing-median baseline (엄격 과거만)
def trailing_lot_baseline(frame, val_col, k=5):
    """각 Lot 대표시각 lot_ts 기준 '직전' k개 Lot 의 median. 현재/미래 Lot 절대 미참조."""
    lot = frame.groupby(C20_COL).agg(lot_ts=("lot_ts", "first"), v=(val_col, "median")).reset_index()
    lot = lot.sort_values("lot_ts").reset_index(drop=True)
    base = np.full(len(lot), np.nan)
    for i in range(len(lot)):
        past = lot.iloc[:i]                                  # i 번째 '이전' 행만
        past = past[past["lot_ts"] < lot.iloc[i]["lot_ts"]]  # 동시각도 배제(엄격)
        if len(past) >= 1:
            base[i] = past["v"].tail(k).median()
    lot["baseline"] = base
    return lot

_probe = trailing_lot_baseline(df, TARGET_COL, k=5)
_viol = 0
for i in range(1, len(_probe)):
    if pd.notna(_probe.iloc[i]["baseline"]):
        cur = _probe.iloc[i]["lot_ts"]; past_max = _probe.iloc[:i]["lot_ts"].max()
        if not (past_max < cur):
            _viol += 1
assert _viol == 0, f"R11 위반: 미래/동시각 참조 {_viol}건"

# 반례 데모: center 창(미래 포함) baseline 은 인과 규약에 걸린다
def _leaky(frame, val_col):
    lot = frame.groupby(C20_COL).agg(lot_ts=("lot_ts", "first"), v=(val_col, "median")).reset_index()
    lot = lot.sort_values("lot_ts").reset_index(drop=True)
    lot["baseline"] = lot["v"].rolling(3, center=True, min_periods=1).median()  # center = 미래 참조
    return lot
_leak = _leaky(df, TARGET_COL); _caught = False
for i in range(1, len(_leak) - 1):
    if _leak.iloc[i + 1]["lot_ts"] >= _leak.iloc[i]["lot_ts"]:
        _caught = True; break
assert _caught, "반례 탐지 실패 — 인과 검사기 점검 필요"
print(f"[셀3b] R11 인과 assert PASS (trailing baseline 미래참조 {_viol}건) | 반례(center) 누수 탐지={_caught}")

[셀3b] R11 인과 assert PASS (trailing baseline 미래참조 0건) | 반례(center) 누수 탐지=True


In [5]:
# [셀4] 확장창 워크포워드 3-arm (인과적) — 벤치마크 self-check
Xl = df[LEAN]; Xc = df[CORE]; lot_ts = df["lot_ts"]
t0 = df["wf_ts"].min(); init_end = t0 + pd.Timedelta(weeks=INIT_WEEKS); end = df["wf_ts"].max()
H = pd.Timedelta(days=H_DAYS)

init_idx = np.where((lot_ts <= init_end).to_numpy())[0]
m_nr = make_xgb(XGB_PARAMS, XGB_ROUNDS); m_nr.fit(Xl.iloc[init_idx], y[init_idx])   # (b) 무재학습 = 1회 학습·고정
print(f"[셀4] 초기창 n={len(init_idx)} (~{str(init_end)[:10]}) | 무재학습 모델 고정")

folds = []; PY = {"lean": [], "nr": [], "core": []}; PP = {"lean": [], "nr": [], "core": []}
t_start = time.time(); T = init_end
while T < end:
    te = ((lot_ts > T) & (lot_ts <= T + H)).to_numpy(); te_idx = np.where(te)[0]
    if len(te_idx) < MIN_TEST:
        T = T + H; continue
    tr_idx = np.where((lot_ts <= T).to_numpy())[0]           # 확장창 (과거만)
    yte = y[te_idx]
    ma = make_xgb(XGB_PARAMS, XGB_ROUNDS); ma.fit(Xl.iloc[tr_idx], y[tr_idx]); pa = ma.predict(Xl.iloc[te_idx])  # (a) 재학습 lean
    mc = make_xgb(XGB_PARAMS, XGB_ROUNDS); mc.fit(Xc.iloc[tr_idx], y[tr_idx]); pc = mc.predict(Xc.iloc[te_idx])  # (c) 재학습 core
    pb = m_nr.predict(Xl.iloc[te_idx])                                                                            # (b) 무재학습
    folds.append(dict(cut=str(T)[:10], n=int(len(te_idx)), train_n=int(len(tr_idx)),
                      retrain_lean=round(_rmse(yte, pa), 3), noretrain=round(_rmse(yte, pb), 3),
                      retrain_core=round(_rmse(yte, pc), 3)))
    for k, p in [("lean", pa), ("nr", pb), ("core", pc)]:
        PY[k].append(yte); PP[k].append(p)
    print(f"  {str(T)[:10]} (test {len(te_idx):>4}, train {len(tr_idx):>5}): "
          f"재학습 {_rmse(yte,pa):6.1f} | 무재학습 {_rmse(yte,pb):6.1f} | core10재 {_rmse(yte,pc):6.1f}")
    T = T + H
print(f"[셀4] 완료: {len(folds)} 주간 fold | {time.time()-t_start:.0f}s")

[셀4] 초기창 n=4661 (~2018-12-29) | 무재학습 모델 고정
  2018-12-29 (test  834, train  4661): 재학습  147.0 | 무재학습  147.0 | core10재  141.8
  2019-01-05 (test 1077, train  5495): 재학습  113.3 | 무재학습  168.3 | core10재  106.2
  2019-01-12 (test 1211, train  6572): 재학습   75.2 | 무재학습  216.0 | core10재   73.0
  2019-01-19 (test 1475, train  7783): 재학습  109.5 | 무재학습  282.7 | core10재  113.5
  2019-01-26 (test 1408, train  9258): 재학습   68.1 | 무재학습  292.4 | core10재   68.9
  2019-02-02 (test 1273, train 10666): 재학습   87.0 | 무재학습  319.1 | core10재   85.7
[셀4] 완료: 6 주간 fold | 5s


In [6]:
# [셀5] pooled 집계 + GV0 벤치마크 self-check (판정은 강건)
def pooled(k):
    return _rmse(np.concatenate(PY[k]), np.concatenate(PP[k]))
p_lean, p_nr, p_core = pooled("lean"), pooled("nr"), pooled("core")
r2_core, r2_lean = r2_honest(p_core), r2_honest(p_lean)

print("=== [셀5] pooled (전 fold 통합) — 벤치마크 self-check ===")
print(f"  재학습 core10  = {p_core:7.3f} (R²={r2_core})   [벤치마크 목표 98.36 / R² 0.85]")
print(f"  재학습 lean-85 = {p_lean:7.3f} (R²={r2_lean})   [REPORT_13 99.84]")
print(f"  무재학습       = {p_nr:7.3f}                     [REPORT_13 254.9]")
print()
d_core = abs(p_core - 98.36)
print(f"  GV0 self-check: core10 |Δvs98.36|={d_core:.2f}  "
      f"| 재학습<무재학습={'OK' if p_core<p_nr else 'NG'}  | 센서>백본={'OK' if p_lean>p_core else 'NG'}")
print("  → 공식 GV0 판정은 강건(R4): core10 ≈98.36·R²≈0.85 재현이면 하니스 신뢰, 아니면 원인규명 전 전진 금지.")

# V1~V3 에서 재사용할 rolling 평가 함수 (신규 피처셋 X_feat 를 이 프레임으로 채점)
def rolling_pooled(X_feat, seeds=(42,)):
    """확장창 rolling-origin time-CV. 신규 피처셋 평가 전용(V2~). 다중시드=이중딥핑 통제(R7)."""
    out = {}
    for sd in seeds:
        PYk, PPk = [], []; Tt = init_end
        while Tt < end:
            te = ((lot_ts > Tt) & (lot_ts <= Tt + H)).to_numpy(); ti = np.where(te)[0]
            if len(ti) < MIN_TEST: Tt = Tt + H; continue
            tri = np.where((lot_ts <= Tt).to_numpy())[0]
            mm = make_xgb(XGB_PARAMS, XGB_ROUNDS); mm.set_params(random_state=sd)
            mm.fit(X_feat.iloc[tri], y[tri]); PYk.append(y[ti]); PPk.append(mm.predict(X_feat.iloc[te]))
            Tt = Tt + H
        rm = _rmse(np.concatenate(PYk), np.concatenate(PPk)); out[sd] = (rm, r2_honest(rm))
    return out

=== [셀5] pooled (전 fold 통합) — 벤치마크 self-check ===
  재학습 core10  =  98.361 (R²=0.8587)   [벤치마크 목표 98.36 / R² 0.85]
  재학습 lean-85 =  99.840 (R²=0.8545)   [REPORT_13 99.84]
  무재학습       = 254.899                     [REPORT_13 254.9]

  GV0 self-check: core10 |Δvs98.36|=0.00  | 재학습<무재학습=OK  | 센서>백본=OK
  → 공식 GV0 판정은 강건(R4): core10 ≈98.36·R²≈0.85 재현이면 하니스 신뢰, 아니면 원인규명 전 전진 금지.


In [7]:
# [셀6] same-era 로더 — valid/test 원시 트레이스 (V0=로드/시각확인만, 채점 V4 보류 · R3)
def load_same_era(split, nrows=None):
    p = None
    for d in ["..", "../문제1(하)", "문제1(하)", "."]:
        cand = os.path.join(d, f"{split}_X.csv")
        if os.path.exists(cand): p = cand; break
    if p is None:
        return {"split": split, "found": False}
    raw = pd.read_csv(p, nrows=nrows)
    ts = pd.to_datetime(raw.get("C40"), format=FMT, errors="coerce")
    return {"split": split, "found": True, "path": p, "cols": int(raw.shape[1]),
            "rows": int(raw.shape[0]), "ts_parse_rate": round(float(ts.notna().mean()), 3),
            "n_wf": int(raw[ID_COL].nunique())}

for s in ["valid", "test"]:
    print(f"[셀6] same-era {s}: {load_same_era(s, nrows=5000)}")
# 정답 폴더는 '경로만' 확인 (R3 — V0~V3 파이프라인 접근 금지, V4 부록 채점 1회 전용)
ANS = next((d for d in ["../문제1_하_answer", "문제1_하_answer", "../../문제1_하_answer"] if os.path.isdir(d)), None)
print(f"[셀6] 정답 폴더 존재={ANS is not None} (R3: 경로만, 미접근) | 집계·채점 파이프라인=V4")

[셀6] same-era valid: {'split': 'valid', 'found': True, 'path': '../문제1(하)\\valid_X.csv', 'cols': 64, 'rows': 5000, 'ts_parse_rate': 1.0, 'n_wf': 484}
[셀6] same-era test: {'split': 'test', 'found': True, 'path': '../문제1(하)\\test_X.csv', 'cols': 66, 'rows': 5000, 'ts_parse_rate': 0.0, 'n_wf': 484}
[셀6] 정답 폴더 존재=True (R3: 경로만, 미접근) | 집계·채점 파이프라인=V4


In [8]:
# [셀7] P1~P4 피처 빌더 스켈레톤 — 계약(contract)만. 생성=V1, 평가=V2.
# 공통 원리: 절대 센서값이 아니라 '상대값'(PM위상/레짐/직전baseline/잔차). floor·누수 규칙 공통(R2·R10·R11).
def build_P1_pm_relative(df, sensor_stat_cols, pm_col="days_since_last_pm", n_bins=5):
    """P1: days_since_last_pm 구간별 센서 z-score. [인과] bin 통계(mean/std)는 rolling 시 과거 fold 만."""
    raise NotImplementedError("V1 구현")

def build_P2_regime_conditional(df, sensor_stat_cols, regime_col="is_high_regime"):
    """P2: is_high_regime 별 센서 baseline 차감. [인과] baseline 은 train fold 통계만."""
    raise NotImplementedError("V1 구현")

def build_P3_stationary(df, sensor_stat_cols, k_lots=5):
    """P3: 직전 k Lot median 대비 편차(detrend). [인과] trailing_lot_baseline() 규약 사용(R11)."""
    raise NotImplementedError("V1 구현")

def build_P4_target_decomp(df, core_cols, sensor_cols):
    """P4: 1단 core10→레짐레벨, 2단 센서→잔차. [인과] 1단 예측은 OOF/과거만(잔차 누수 방지)."""
    raise NotImplementedError("V1 구현")

print("[셀7] P1~P4 스켈레톤 로드 OK (생성=V1) | 인과규약=trailing_lot_baseline() | rolling평가=rolling_pooled()")
print("\n=== V0 하니스 self-check 완료 — GV0 판정 대기(강건) ===")

[셀7] P1~P4 스켈레톤 로드 OK (생성=V1) | 인과규약=trailing_lot_baseline() | rolling평가=rolling_pooled()

=== V0 하니스 self-check 완료 — GV0 판정 대기(강건) ===
